### DuckDB Basics

#### Overview

In this section, we will explore cloud-native vector datasets and use DuckDB to query and locate data directly from cloud. We also use [Lonboard](https://developmentseed.org/lonboard/latest/) for interactively visualizing query results.

#### Overview of the Task

We will query and load an Administrative Boundaries dataset provided by FieldMaps as a cloud-native GeoParquet file. We will interactively query and save the required boundary polygon for our analysis.

In [1]:
import duckdb
import geopandas as gpd
from lonboard import viz
import os

#### DuckDB

DuckDB is a modern high-performance database engine that allows querying large files easily. It has built-in support for spatial data and can be used to query large remote spatial files without downloading it first.

We initialize DuckDB and enable the spatial extension.

In [2]:
conn = duckdb.connect()
conn.install_extension('spatial')
conn.load_extension('spatial')

#### Query Remote Dataset

[FieldMaps](https://fieldmaps.io/data/) provides open datasets of global administrative boundaries from multiple providers. We will query the `admin2` boundaries from GeoBoundaries in the GeoParquet format. The source data is a 2 GB `.parquet` file containing 48,000+ polygons. Instead of downloading this, we can query it and extract just the subset we require.

In [3]:
parquet_url = 'https://data.fieldmaps.io/edge-matched/open/intl/adm2_polygons.parquet'

DuckDB supports standard SQL syntax for querying. Let's check some basic information about the dataset.

In [4]:
query = f'''
SELECT COUNT(*) FROM read_parquet('{parquet_url}')
'''

result = conn.sql(query).fetchone()
print('Total Features:', result)

Total Features: (48671,)


We can use `DESCRIBE` clause to get the available columns. We can turn the results of any query to a DataFrame using the `.df()`.

In [5]:
query = f'''
DESCRIBE SELECT * FROM read_parquet('{parquet_url}')
'''

columns = conn.sql(query).df()
columns

,column_name,column_type,null,key,default,extra
0,fid,BIGINT,YES,None,None,None
1,adm2_id,VARCHAR,YES,None,None,None
2,adm2_src,VARCHAR,YES,None,None,None
3,adm2_name,VARCHAR,YES,None,None,None
4,adm2_name1,VARCHAR,YES,None,None,None
5,adm2_name2,VARCHAR,YES,None,None,None
6,adm1_id,VARCHAR,YES,None,None,None
7,adm1_src,VARCHAR,YES,None,None,None
8,adm1_name,VARCHAR,YES,None,None,None
9,adm1_name1,VARCHAR,YES,None,None,None


We can now form a query to find all `admin1 names` (states/provinces) in a specific country. We will use this in the next step to fetch all `admin2` regions within a specific `admin1` area. The `adm0_src` uses the 3-digit [ISO code](https://en.wikipedia.org/wiki/ISO_3166-1_alpha-3) for each country.

In [6]:
country = 'PHL'

query = f'''
SELECT DISTINCT adm1_name
FROM read_parquet('{parquet_url}')
WHERE adm0_src = '{country}'
ORDER BY adm1_name
'''

df_admin1 = conn.sql(query).df()
df_admin1

,adm1_name
0,ARMM
1,Bicol Region
2,CAR
3,Cagayan Valley
4,Calabarzon
5,Caraga
6,Central Luzon
7,Central Visayas
8,Davao Region
9,Eastern Visayas


Notice that the dataset has a `geometry` column, which stores the layer geometry. We can now query for all `admin2` polygons within our chosen `admin1` region. Also, since Parquet is a columnar format, it is efficient to fetch only the required columns.

In [7]:
adm1_name = 'Calabarzon'

query = f'''
SELECT adm1_name, adm1_id, adm2_name, adm2_id, ST_AsWKB(geometry) AS geometry
FROM read_parquet('{parquet_url}')
WHERE
    adm0_src = '{country}' and
    adm1_name = '{adm1_name}'
'''

df_admin2 = conn.sql(query).df()
df_admin2

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,adm1_name,adm1_id,adm2_name,adm2_id,geometry
0,Calabarzon,PHL-20230119-05,Batangas,PHL-20230119-05-1,"[1, 6, 0, 0, 0, 55, 0, 0, 0, 1, 3, 0, 0, 0, 1,..."
1,Calabarzon,PHL-20230119-05,Cavite,PHL-20230119-05-2,"[1, 6, 0, 0, 0, 15, 0, 0, 0, 1, 3, 0, 0, 0, 1,..."
2,Calabarzon,PHL-20230119-05,Laguna,PHL-20230119-05-3,"[1, 6, 0, 0, 0, 1, 0, 0, 0, 1, 3, 0, 0, 0, 1, ..."
3,Calabarzon,PHL-20230119-05,Quezon,PHL-20230119-05-4,"[1, 6, 0, 0, 0, 161, 0, 0, 0, 1, 3, 0, 0, 0, 1..."
4,Calabarzon,PHL-20230119-05,Rizal,PHL-20230119-05-5,"[1, 6, 0, 0, 0, 1, 0, 0, 0, 1, 3, 0, 0, 0, 1, ..."


We turn the results into a GeoPandas GeoDataFrame by specifying the geometry column and the CRS.

In [8]:
gdf_admin2 = gpd.GeoDataFrame(
    df_admin2,
    geometry=gpd.GeoSeries.from_wkb(df_admin2['geometry'].apply(bytes)),
    crs='EPSG:4326'
)

gdf_admin2

,adm1_name,adm1_id,adm2_name,adm2_id,geometry
0,Calabarzon,PHL-20230119-05,Batangas,PHL-20230119-05-1,"MULTIPOLYGON (((120.49154 14.05415, 120.4915 1..."
1,Calabarzon,PHL-20230119-05,Cavite,PHL-20230119-05-2,"MULTIPOLYGON (((120.89257 14.08749, 120.89272 ..."
2,Calabarzon,PHL-20230119-05,Laguna,PHL-20230119-05-3,"MULTIPOLYGON (((121.22897 14.0029, 121.23189 1..."
3,Calabarzon,PHL-20230119-05,Quezon,PHL-20230119-05-4,"MULTIPOLYGON (((121.78685 13.91013, 121.78662 ..."
4,Calabarzon,PHL-20230119-05,Rizal,PHL-20230119-05-5,"MULTIPOLYGON (((121.24366 14.24262, 121.25342 ..."


In [9]:
viz(gdf_admin2)

In [10]:
viz?

Signature:
viz(
    data: 'VizDataInput | list[VizDataInput] | tuple[VizDataInput, ...]',
    *,
    scatterplot_kwargs: 'ScatterplotLayerKwargs | None' = None,
    path_kwargs: 'PathLayerKwargs | None' = None,
    polygon_kwargs: 'PolygonLayerKwargs | None' = None,
    map_kwargs: 'MapKwargs | None' = None,
) -> 'Map'
Docstring:
Plot your data easily.

The goal of this function is to make it simple to get _something_ showing on a map.
For more control over rendering, construct [`Map`][lonboard.Map] and
[`Layer`][lonboard.BaseLayer] objects directly.

This function accepts a variety of geospatial inputs:

- GeoPandas [`GeoDataFrame`][geopandas.GeoDataFrame].
- GeoPandas [`GeoSeries`][geopandas.GeoSeries].
- numpy array of Shapely objects.
- Single Shapely object.
- A DuckDB query with a spatial column from DuckDB Spatial.

    !!! info

        The DuckDB query must be run with
        [`duckdb.sql()`](https://duckdb.org/docs/api/python/reference/#duckdb.sql)
        or
        [`duckd

#### Save the Results

We can save the selected subset as a GeoPackage. We can now save it as a file.

In [14]:
fname_output = 'admin2.gpkg'
dir_output = 'output'

fp_output =  os.path.join(dir_output, fname_output)
gdf_admin2.to_file(fp_output)
print(f'Saved to {fp_output}')

Saved to output\admin2.gpkg


#### Exercise
[Overture Maps](https://overturemaps.org/) provides free and open map data curated from sources like OpenStreetMap. The entire dataset is available in cloud-native GeoParquet format.

Extract the boundary for your selected city and save it to your output directory in GeoJSON format as `aoi.geojson`.

##### Tips

- Search for your city/region of interest using [Overture Explorer](https://explore.overturemaps.org/) and replace the `country_iso2`, `city_name`, and `region` variables with the appropriate values.
- Cities are not uniformly represented across the world. Some cities are tagged as _locality_, while others with _county_ or _localadmin_. The SQL query below tries to capture all the variations, but if you get no matches, you can relax the query by commenting out some lines via prefixing it with `--`.
- By default, the boundary tagged as `locality` will be picked. To see other options, comment the line starting with `LIMIT 1`.

In [15]:
country_iso2 = 'PH'
city_name = 'Antipolo'
region = 'PH-40'

In [16]:
# Overture data does monthly releases of their dataset
# Find the latest release at https://stac.overturemaps.org/
OVERTURE_RELEASE = '2026-06-17.0'

s3_path = (
    f's3://overturemaps-us-west-2/release/{OVERTURE_RELEASE}/'
    'theme=divisions/type=division_area/*'
)

query = f"""
SELECT
    id,
    names.primary AS primary_name,
    names.common.en AS common_name,
    subtype,
    country,
    region,
    ST_AsWKB(geometry) AS geometry,
FROM read_parquet(
    '{s3_path}',
    filename=true,
    hive_partitioning=1
)
WHERE 
    subtype in ('locality', 'country', 'localadmin', 'region') AND
    country = '{country_iso2}' AND
    region = '{region}' AND
    (names.primary ILIKE '{city_name}' OR names.common.en ILIKE '{city_name}') AND
    is_land = true      -- exclude maritime extensions
ORDER BY
    -- prefer 'locality' over other types
    CASE subtype WHEN 'locality' THEN 0 ELSE 1 END
LIMIT 1
"""

results = conn.sql(query).df()
results

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,id,primary_name,common_name,subtype,country,region,geometry
0,ca935ecc-a0ca-4442-8f17-f6a6fa856973,Antipolo,Antipolo,locality,PH,PH-40,"[1, 3, 0, 0, 0, 1, 0, 0, 0, 55, 5, 0, 0, 121, ..."


In [17]:
gdf_aoi = gpd.GeoDataFrame(
    results,
    geometry=gpd.GeoSeries.from_wkb(results['geometry'].apply(bytes)),
    crs='EPSG:4326'
)

viz(gdf_aoi)

In [18]:
fp_aoi = os.path.join(dir_output, 'aoi.geojson')
gdf_aoi.to_file(fp_aoi)